In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

In [2]:
# phase velocity and group velocity
samples = 25
x = np.linspace(-5, 5, 1000)
t_list = np.linspace(0, 5, samples, endpoint=False)
omega = 4 * np.pi

wave_col = 'greenyellow'
phase_col = 'crimson'
group_col = 'dodgerblue'

def build_fronts(x_pos, offset, n_fronts):
    front_list = []
    y_list = []
    for n in range(n_fronts):
        front_list.append(x_pos+n*offset)
        front_list.append(x_pos+n*offset)
        y_list.append((-1)**n*2)
        y_list.append((-1)**(n+1)*2)
    return front_list, y_list

wave_funcs = [
    lambda x, t, g: g * np.sin(omega*(x-t)),
    lambda x, t, g: g * np.sin(omega*(x+t)),
    lambda x, t, g: g * np.sin(omega*(x-2*t)),
    lambda x, t, g: g * np.sin(omega*(x-t/2)),
    lambda x, t, g: g * np.sin(omega*x),
    lambda x, t, g: np.exp(-x**2) * np.sin(omega*(x-t/2))
]

p_vel_funcs = [
    lambda t: t + 1/8,
    lambda t: -t + 1/8,
    lambda t: 2*t + 1/8,
    lambda t: t/2 + 1/8,
    lambda t: 1/8,
    lambda t: t/2 + 1/8
]

x_offs = [x+5*n for n in range(-1, 3)]
waves = [[] for _ in range(6)]

for t in t_list:
    sums = [0]*6
    
    for x_off in x_offs:
        group = np.exp(-((x_off-t)**2))
                       
        for i, f in enumerate(wave_funcs):
            sums[i] += f(x_off, t, group)

    for wave, total in zip(waves, sums):
        wave.append(total)

plt.style.use('dark_background')
fig, ax = plt.subplots(nrows=3, ncols=2, figsize=(8,6))

fig.subplots_adjust(
    left=0.025, right=0.975,
    bottom=0.03, top=0.86,
    wspace=0.1, hspace=0.3
)

fig.suptitle('Phase Velocity and Group Velocity', fontsize=20)

for row in (0,1):
    for col in (0,1):
        ax[row][col].text(-0.5, 1.32, 'phase vel.', color=phase_col, ha='right', fontsize=15)
        ax[row][col].text(0.5, 1.32, 'group vel.', color=group_col, ha='left', fontsize=15)

ax[0][0].text(0, 1.32, '=', ha='center',fontsize=15)
ax[0][1].text(0, 1.32, '= -', ha='center',fontsize=15)
ax[1][0].text(0, 1.32, '>', ha='center',fontsize=15)
ax[1][1].text(0, 1.32, '<', ha='center',fontsize=15)

ax[2][0].text(1.05, 1.32, 'phase vel.', color=phase_col, ha='right',fontsize=15)
ax[2][0].text(1.05, 1.32, ' = 0', ha='left',fontsize=15)
ax[2][1].text(0.95, 1.32, 'group vel.', color=group_col, ha='right',fontsize=15)
ax[2][1].text(0.95, 1.32, ' = 0', ha='left',fontsize=15)

y_list = []
p_list = []
g_list = []

for row in range(3):
    for col in range(2):
        ax[row][col].set_xticks([])
        ax[row][col].set_yticks([])

        ax[row][col].set_xlim(-5, 5)
        ax[row][col].set_ylim(-1.2, 1.2)

        y, = ax[row][col].plot([], [], color=wave_col, alpha=0.8)
        y_list.append(y)

        p, = ax[row][col].plot([], [], color=phase_col)
        p_list.append(p)
        
        g, = ax[row][col].plot([], [], color=group_col)
        g_list.append(g)

def update(frame):

    for n, y, p, g in zip(range(6), y_list, p_list, g_list):

        t = t_list[frame]
        
        y.set_data(x, waves[n][frame])

        p_pos = p_vel_funcs[n](t)
        p_fronts, p_heights = build_fronts(p_pos-15, 2.5, 10)
        p.set_data(p_fronts, p_heights)

        g_pos = t if n != 5 else 0
        g_fronts, g_heights = build_fronts(g_pos-8.75, 5, 3)
        g.set_data(g_fronts, g_heights)

    return *y_list, *p_list, *g_list

anim = FuncAnimation(fig, update, frames=samples, blit=True)
# anim.save(f'phase_group.gif', writer=PillowWriter(fps=20))
plt.close(fig)
HTML(anim.to_jshtml())